In [1]:
"""
build_discovery_sample.py

Builds a stratified sample of top-level comments for LLooM concept
DISCOVERY (i.e., "what kinds of responses/topics show up in the burnout
comment community" -- not scoring against a fixed taxonomy).

Unlike the 150-row pilot (10 posts, 3 subreddits), this sample is built to
be genuinely representative:
  - stratified across all 5 subreddits (so sysadmin, ~81% of posts, doesn't
    dominate the discovered topics)
  - stratified across bat_score (1-4) and/or MD/EMO construct
  - drawn from many distinct posts, not a few large threads

Reads:
    master_comments_filtered.csv   (296,442 top-level comments)
    bat_score_pos.csv              (12,237 posts: post_id, bat_score,
                                     subreddit, MD/EMO construct)

Writes:
    discovery_sample.csv   -- ready to feed into LLooM's l.gen()

Usage:
    python build_discovery_sample.py
"""

import pandas as pd
import numpy as np

# ── CONFIG ───────────────────────────────────────────────────────────────
DATA_DIR = '/Users/nadia/Desktop/redditRun_june/comment_data/'
COMMENTS_FILE = DATA_DIR + 'master_comments_filtered.csv'
BAT_FILE = DATA_DIR + 'bat_score_pos.csv'
OUTPUT_FILE = DATA_DIR + 'discovery_sample.csv'

TARGET_SAMPLE_SIZE = 800     # total comments in the final sample
MIN_WORD_COUNT = 5           # drop near-empty comments (e.g. "this", "lol")
MAX_PER_POST = 15            # cap comments drawn from any single post, so
                              # one large thread can't dominate the topics
MIN_PER_STRATUM = 5          # floor -- try to get at least this many per
                              # subreddit x bat_score cell if available
RANDOM_SEED = 42

# Subreddit lives on the COMMENTS file (subreddit_source), not on
# bat_score_pos.csv -- confirmed from actual column inspection.
COMMENTS_SUBREDDIT_COL = 'subreddit_source'

# bat_score_pos.csv is post-level only (row_type is always 'post').
# EX/EMO/COG/MD are independent YES/NO flags, not one categorical label --
# a post can be MD=YES and EMO=YES at the same time. For THIS run (topic
# discovery across subreddits), we stratify by subreddit x bat_score only.
# To additionally stratify by a specific construct later (e.g. for the
# MD-vs-EMO composition analysis), set this to one of: 'EX','EMO','COG','MD'
BAT_CONSTRUCT_COL = None


def inspect_inputs():
    comments = pd.read_csv(COMMENTS_FILE, dtype={'id': str, 'post_id': str}, low_memory=False)
    bat = pd.read_csv(BAT_FILE, dtype={'post_id': str})
    print(f"comments: {len(comments):,} rows, columns: {list(comments.columns)}")
    print(f"bat_score_pos: {len(bat):,} rows, columns: {list(bat.columns)}")
    return comments, bat


def build_sample():
    comments = pd.read_csv(COMMENTS_FILE, dtype={'id': str, 'post_id': str}, low_memory=False)
    bat = pd.read_csv(BAT_FILE, dtype={'post_id': str})

    if COMMENTS_SUBREDDIT_COL not in comments.columns:
        raise ValueError(
            f"'{COMMENTS_SUBREDDIT_COL}' not found in master_comments_filtered.csv. "
            f"Available columns: {list(comments.columns)}."
        )

    bat_cols = ['post_id', 'bat_score']
    if BAT_CONSTRUCT_COL:
        if BAT_CONSTRUCT_COL not in bat.columns:
            raise ValueError(
                f"'{BAT_CONSTRUCT_COL}' not found in bat_score_pos.csv. "
                f"Available columns: {list(bat.columns)}."
            )
        bat_cols.append(BAT_CONSTRUCT_COL)

    bat_small = bat[bat_cols].drop_duplicates(subset='post_id')

    # Join comments -> post-level bat_score (subreddit already lives on
    # the comments file, so no need to pull it from bat_small)
    df = comments.merge(bat_small, on='post_id', how='inner')
    print(f"Comments joined to bat_score>0 posts: {len(df):,} "
          f"(of {len(comments):,} total comments)")

    # Clean: drop empty/near-empty comments
    df = df[df['cleaned_body'].notna()]
    df = df[df['cleaned_body'].astype(str).str.strip().str.len() > 0]
    if 'word_count' in df.columns:
        df = df[df['word_count'] >= MIN_WORD_COUNT]
    print(f"After empty/short-comment filtering: {len(df):,}")

    # Cap comments per post so one large thread can't dominate
    df = (
        df.groupby('post_id', group_keys=False)
        .apply(lambda g: g.sample(min(len(g), MAX_PER_POST), random_state=RANDOM_SEED))
    )
    print(f"After per-post cap ({MAX_PER_POST} max): {len(df):,}")

    # Build strata: subreddit x bat_score (add construct too if configured)
    strata_cols = [COMMENTS_SUBREDDIT_COL, 'bat_score']
    if BAT_CONSTRUCT_COL:
        strata_cols.append(BAT_CONSTRUCT_COL)

    strata_sizes = df.groupby(strata_cols).size().reset_index(name='n_available')
    print(f"\n{len(strata_sizes)} strata found ({' x '.join(strata_cols)}):")
    print(strata_sizes.to_string(index=False))

    n_strata = len(strata_sizes)
    target_per_stratum = max(MIN_PER_STRATUM, TARGET_SAMPLE_SIZE // n_strata)

    sampled_parts = []
    for _, row in strata_sizes.iterrows():
        mask = np.ones(len(df), dtype=bool)
        for col in strata_cols:
            mask &= (df[col] == row[col])
        stratum_df = df[mask]
        n_take = min(len(stratum_df), target_per_stratum)
        sampled_parts.append(stratum_df.sample(n_take, random_state=RANDOM_SEED))

    sample = pd.concat(sampled_parts, ignore_index=True)

    # If we're under target (small strata capped us out), top up randomly
    # from whatever's left, respecting the per-post cap already applied.
    if len(sample) < TARGET_SAMPLE_SIZE:
        remaining = df[~df['id'].isin(sample['id'])]
        n_topup = min(len(remaining), TARGET_SAMPLE_SIZE - len(sample))
        if n_topup > 0:
            topup = remaining.sample(n_topup, random_state=RANDOM_SEED)
            sample = pd.concat([sample, topup], ignore_index=True)

    print(f"\nFinal sample size: {len(sample):,}")
    print(f"Posts represented: {sample['post_id'].nunique():,}")
    print(f"\nSubreddit distribution:\n{sample[COMMENTS_SUBREDDIT_COL].value_counts()}")
    print(f"\nbat_score distribution:\n{sample['bat_score'].value_counts().sort_index()}")
    if BAT_CONSTRUCT_COL:
        print(f"\n{BAT_CONSTRUCT_COL} distribution:\n{sample[BAT_CONSTRUCT_COL].value_counts()}")

    sample.to_csv(OUTPUT_FILE, index=False)
    print(f"\n✓ Saved: {OUTPUT_FILE}")
    return sample


if __name__ == '__main__':
    # First run: uncomment the next two lines ONLY to check column names,
    # then set COMMENTS_SUBREDDIT_COL / BAT_CONSTRUCT_COL above and comment
    # them back out.
    # inspect_inputs()
    # raise SystemExit("Update column names above, then rerun.")

    build_sample()

Comments joined to bat_score>0 posts: 296,442 (of 296,442 total comments)
After empty/short-comment filtering: 296,442


/var/folders/41/b55bchyx62j34hmsgfhfk2gw0000gp/T/ipykernel_16782/1149092002.py:101: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby('post_id', group_keys=False)


After per-post cap (15 max): 106,118

19 strata found (subreddit_source x bat_score):
    subreddit_source  bat_score  n_available
SecurityCareerAdvice          1         1312
SecurityCareerAdvice          2          586
SecurityCareerAdvice          3          375
SecurityCareerAdvice          4           64
           asknetsec          1          761
           asknetsec          2          301
           asknetsec          3          183
           asknetsec          4           61
                ciso          1           64
                ciso          2           45
                ciso          3           35
       cybersecurity          1         6390
       cybersecurity          2         3404
       cybersecurity          3         2050
       cybersecurity          4          505
            sysadmin          1        51007
            sysadmin          2        23343
            sysadmin          3        12549
            sysadmin          4         3083

Final sample 